In [2]:
from statsbombpy import sb
import os 

In [3]:
import os
import glob
import pandas as pd
from statsbombpy import sb
from tqdm import tqdm

In [4]:
competitions = sb.competitions()

/Users/antonio/xgPredictor/.venv/lib/python3.13/site-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


In [16]:
competitions[competitions["competition_name"] == "Premier League"]

,competition_id,season_id,country_name,competition_name,competition_gender,competition_youth,competition_international,season_name,match_updated,match_updated_360,match_available_360,match_available
64,2,27,England,Premier League,male,False,False,2015/2016,2025-04-23T14:36:29.347042,2021-06-13T16:17:31.694,NaN,2025-04-23T14:36:29.347042
65,2,44,England,Premier League,male,False,False,2003/2004,2025-06-24T13:53:07.585114,2021-06-13T16:17:31.694,NaN,2025-06-24T13:53:07.585114


In [17]:
matches = sb.matches(competition_id=2, season_id=27)

/Users/antonio/xgPredictor/.venv/lib/python3.13/site-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


In [ ]:
def fetch_and_save_shots(matches_df, output_path="data/bronze/shots"):
    #just to be sure that the dir exists
    os.makedirs(output_path, exist_ok=True)
    
    match_ids = matches_df["match_id"].unique()
    
    for m_id in tqdm(match_ids, desc="Ingesting Matches"):

        file_checkpoint = f"{output_path}/match_{m_id}.pkl"
        
        if os.path.exists(file_checkpoint):
            continue
            
        try:
            events = sb.events(match_id=m_id)
            if "type" in events.columns:
                shot_events = events[events["type"] == "Shot"].copy()
                
                if not shot_events.empty:
                    shot_events.to_pickle(file_checkpoint)

        except Exception as e:
            print(f"Error processing match {m_id}: {e}")

# Execution
fetch_and_save_shots(matches)


Ingesting Matches:   0%|          | 0/380 [00:00<?, ?it/s]/Users/antonio/xgPredictor/.venv/lib/python3.13/site-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
Ingesting Matches:   0%|          | 1/380 [00:00<00:53,  7.07it/s]/Users/antonio/xgPredictor/.venv/lib/python3.13/site-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
Ingesting Matches:   1%|          | 2/380 [00:00<00:46,  8.10it/s]/Users/antonio/xgPredictor/.venv/lib/python3.13/site-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
Ingesting Matches:   1%|          | 3/380 [00:00<00:45,  8.26it/s]/Users/antonio/xgPredictor/.venv/lib/python3.13/site-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
Ingesting Matches:   1%|          | 

In [5]:
def load_bronze_layer(path="data/bronze/shotsByMatches/"):
    # Glob pusca apenas arquivos da extensão
    files = glob.glob(f"{path}/*.pkl")
    if not files:
        print("No pickle files found in the directory.")
        return pd.DataFrame()
    
    return pd.concat([pd.read_pickle(f) for f in files], ignore_index=True)


In [ ]:
df_shots = load_bronze_layer()
csv_bronze_path = "data/bronze/csv"

df_shots.to_csv(f"{csv_bronze_path}/statsbomb_shots_final.csv", index=False)


## To do -> Arrumar caminhos